#### Produce the Thermal-UAV dataset

In [ ]:
#!/usr/bin/env python3
"""
DJI Image Processing Script - Supports new data directory structure
Features:
1. Read JPG images from all DJI folders
2. Extract EXIF information (latitude, longitude, altitude, attitude angles)
3. Organize data directory structure according to the new format
4. Generate CSV information files

New directory structure:
Thermal-UAV
    ├── train/test/valid
    │   └── {city}
    │       └── {data_type}
    │           ├── Thermal/
    │           ├── Visible/
    │           ├── Thermal_info.csv
    │           └── Visible_info.csv
"""

import os
import shutil
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS
import json
import re
import csv

def extract_dji_exif(image_path: Path):
    """
    Read JPG in binary mode, extract 'drone-dji:' fields and standard EXIF info from the rdf section.
    """
    # RDF head and tail markers
    RDF_HEAD = b'<rdf:Description '
    RDF_TAIL = b'</rdf:Description>'

    try:
        with image_path.open('rb') as f:
            data = bytearray()
            flag = False
            for line in f:          # Read line by line, performance is sufficient
                if RDF_HEAD in line:
                    flag = True
                if flag:
                    data += line
                if RDF_TAIL in line:
                    break

        if not data:                # No RDF block found
            return None

        # Convert to string, keeping only ascii part
        text = data.decode('ascii', errors='ignore')

        # Collect 'drone-dji:' lines
        lines = [L.strip() for L in text.splitlines()
                 if L.strip().startswith('drone-dji:')]

        dji = {}
        for raw in lines:
            # Remove prefix "drone-dji:"
            kv = raw[10:]               
            if '=' not in kv:
                continue
            k, v = kv.split('=', 1)     # Split only once
            # Remove quotes from v
            v = v.strip(' "\'')
            # Convert to float
            try:
                dji[k.strip()] = float(v)
            except ValueError:
                dji[k.strip()] = v
        
        # Extract shooting date and time
        try:
            with Image.open(image_path) as img:
                exif = img._getexif()
                if exif:
                    # Tag ID for DateTimeOriginal is 36867
                    datetime_original = exif.get(36867)  
                    if datetime_original:
                        dji['DateTimeOriginal'] = datetime_original
        except Exception as e:
            print(f'Failed to read standard EXIF for {image_path}: {e}')

        # Uniformly output core fields
        return {
            'GpsLatitude':       dji.get('GpsLatitude',      0.0), 
            'GpsLongitude':      dji.get('GpsLongitude',     0.0),
            'AbsoluteAltitude':  dji.get('AbsoluteAltitude', 0.0), # Altitude relative to the ellipsoid surface
            'RelativeAltitude':  dji.get ('RelativeAltitude', 0.0), # The height relative to the take-off point
            'LRFTargetDistance': dji.get('LRFTargetDistance', 0.0), # The straight-line distance from the laser spot to the aircraft.
            'FlightRollDegree':  dji.get('FlightRollDegree', 0.0),
            'FlightPitchDegree': dji.get('FlightPitchDegree',0.0),
            'FlightYawDegree':   dji.get('FlightYawDegree',  0.0),
            'GimbalRollDegree':  dji.get('GimbalRollDegree', 0.0),
            'GimbalPitchDegree': dji.get('GimbalPitchDegree',0.0),
            'GimbalYawDegree':   dji.get('GimbalYawDegree',  0.0),
            'TargetLatitude':    dji.get('LRFTargetLat', 0.0), # The latitude of the laser spot center.
            'TargetLongitude':   dji.get('LRFTargetLon', 0.0), # The longitude of the laser spot center.
            'DateTimeOriginal':  dji.get('DateTimeOriginal', '')
        }

    except Exception as e:
        print(f'Failed to read EXIF for {image_path}: {e}')
        return None

def parse_folder_name(folder_name):
    """
    Parse folder name to extract train/test/valid split, city, and data type.
    Format: DJI_xxx_xxx_train_changsha_300m_ortho_daytime
    
    Returns: (split_type, city, data_type)
        split_type: train/test/valid
        city: City name
        data_type: Data type
    """
        
    # Use regex to match key parts
    # Match format: xxx_xxx_splitType_city_dataType
    pattern = r'_(train|test|valid)_(.+?)_(.+?)$'
    match = re.search(pattern, folder_name)
    
    if match:
        split_type = match.group(1)
        city = match.group(2)
        data_type = match.group(3)
        
        return split_type, city, data_type
    
    # Return default values if parsing fails
    return 'train', 'UnknownCity', 'UnknownType'

def format_date_to_iso(date_str):
    """
    Convert EXIF date format (YYYY:MM:DD HH:MM:SS) to ISO format (YYYY-MM-DDTHH:MM:SS)
    """
    if not date_str:
        return ''
    try:
        # Replace colons with dashes, space with T
        return date_str.replace(':', '-', 2).replace(' ', 'T')
    except Exception:
        return date_str

def write_csv_output(output_path, data_list, file_type='Visible'):
    """
    Write image information to a CSV file
    """
    csv_filename = f"{file_type}_info.csv"
    csv_path = output_path / csv_filename
    
    with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['num', 'city','filename', 'date', 'lat', 'lon', 'abs_height', 'rel_height', 'pitch', 'roll', 'yaw', 'TargetLatitude', 'TargetLongitude']
        writer = csv.writer(csvfile, delimiter=',')  # Use comma separator (standard CSV format)
        
        # Write header
        writer.writerow(fieldnames)
        
        # Write data
        for i, info in enumerate(data_list, 1):
            
            row = [
                i,  # num
                info['city'],  # city
                info['new_filename'],  # filename
                format_date_to_iso(info['exif'].get('DateTimeOriginal', '')),  # date
                info['exif'].get('GpsLatitude', 0.0),  # lat
                info['exif'].get('GpsLongitude', 0.0),  # lon
                info['exif'].get('AbsoluteAltitude', 0.0),  # AbsoluteAltitude used as UAV height GT 
                info['exif'].get('LRFTargetDistance', 0.0),  # LRFTargetDistance used as height above ground to calculate FoV
                # info['exif'].get('RelativeAltitude', 0.0),  # Relative altitude from takeoff point
                info['exif'].get('GimbalPitchDegree', 0.0),  
                info['exif'].get('GimbalRollDegree', 0.0),  
                # info['exif'].get('GimbalYawDegree', 0.0), 
                info['exif'].get('FlightYawDegree', 0.0), 
                info['exif'].get('TargetLatitude', 0.0),  # Target center latitude
                info['exif'].get('TargetLongitude', 0.0),  # Target center longitude
            ]
            writer.writerow(row)
    
    print(f"{file_type} information saved to CSV: {csv_path}")

def process_dji_folders(base_path, output_dir):
    """
    Process all DJI folders and organize the data directory structure according to the new format.
    
    New directory structure:
    Thermal-UAV
        ├── train/test/valid
        │   └── {city}
        │       └── {data_type}
        │           ├── Thermal/
        │           ├── Visible/
        │           ├── Thermal_info.csv
        │           └── Visible_info.csv
    """
    base_path = Path(base_path)
    raw_data_path = base_path / 'DCIM'
    # Create main output folder
    output_main = base_path / output_dir
    output_main.mkdir(exist_ok=True)
    
    # Find all DJI folders (folders starting with 'DJI_')
    dji_folders = []
    for item in raw_data_path.iterdir():
        if item.is_dir() and item.name.startswith('DJI_'):
            dji_folders.append(item)
    
    if not dji_folders:
        print("No folders starting with 'DJI_' found")
        return
    
    print(f"Found {len(dji_folders)} DJI folders")
    
    # Store information for each data directory
    # Structure: {(split_type, city, data_type, img_type): [info_dicts]}
    data_dict = {}
    
    # Process each folder
    for folder in dji_folders:
        
        # Parse folder name
        split_type, city, data_type = parse_folder_name(folder.name)
        
        # Get all JPG images
        jpg_files = list(folder.glob("*.JPG"))
        
        for img_path in jpg_files:
            img_name = img_path.name
            
            # Determine if it is visible light or thermal imaging
            if '_V.JPG' in img_name:
                img_type = 'Visible'
            elif '_T.JPG' in img_name:
                img_type = 'Thermal'
            else:
                continue
            
            # Extract EXIF information
            exif_data = extract_dji_exif(img_path)
            
            if exif_data:
                # Generate new filename: keep original name (including source folder info)
                new_name = f"{folder.name}_{img_name}"
                
                # Record information
                csv_info = {
                    'new_filename': new_name,
                    'city': city,
                    'exif': exif_data
                }
                
                # Store using composite key
                key = (split_type, city, data_type, img_type)
                if key not in data_dict:
                    data_dict[key] = []
                data_dict[key].append(csv_info)
    
    # Create directory structure, copy images, and generate CSV
    print("\nCreating directory structure and generating files...")
    
    # Process by grouped keys
    for (split_type, city, data_type, img_type), info_list in data_dict.items():
        # Create directory paths
        split_dir = output_main / split_type
        city_dir = split_dir / city
        data_type_dir = city_dir / data_type
        
        thermal_dir = data_type_dir / 'Thermal'
        visible_dir = data_type_dir / 'Visible'
        
        thermal_dir.mkdir(parents=True, exist_ok=True)
        visible_dir.mkdir(parents=True, exist_ok=True)
        
        # Select target directory
        if img_type == 'Thermal':
            target_folder = thermal_dir
        else:
            target_folder = visible_dir
        
        # Copy images to target directory
        for info in info_list:
            new_filename = info['new_filename']
            
            # Try to find the original image in the source folder
            source_path = raw_data_path / "_".join(new_filename.split('_')[:-4]) / "_".join(new_filename.split('_')[-4:])
            if source_path.exists():
                shutil.copy2(source_path, target_folder / new_filename)
        
        # Generate CSV file
        write_csv_output(data_type_dir, info_list, img_type)
        
        print(f"  {split_type}/{city}/{data_type}/{img_type}: {len(info_list)} images")
    
    print(f"\nProcessing complete!")
    print(f"Output directory: {output_main}")
    
    # Print directory structure overview
    split_cnt = {}
    print("\nDirectory Structure Overview:")
    for key in sorted(data_dict.keys(), key=lambda x: (x[0], x[1], x[2], x[3])):
        split_type, city, data_type, img_type = key
        count = len(data_dict[key])
        print(f"  {split_type}/{city}/{data_type}/{img_type}/: {count} images")
        
        split_cnt[split_type] = split_cnt.get(split_type, 0) + count
    
    print("\nSplit Set Statistics:")
    for split_type, cnt in split_cnt.items():
        print(f"  {split_type}: {cnt} images")

if __name__ == "__main__":
    # Get current script directory
    current_dir = Path(__file__).parent
    output_dir = "Thermal-UAV"
    print(f"Current working directory: {current_dir}")
    print(f"Output directory: {output_dir}")
    
    # Process DJI folders
    process_dji_folders(current_dir, output_dir)

#### Crop DSM to the same size as the Satellite Image

In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
import json

def crop_dsm_to_sat_extent(dsm_path, sat_path, output_path):
    """
        Crop the DSM to the same extent as the satellite image.
    """
    
    with rasterio.open(sat_path) as sat_src:
        sat_crs = sat_src.crs
        sat_bounds = sat_src.bounds
        print(f"Satellite image CRS: {sat_crs}")
        print(f"Satellite image bounds: {sat_bounds}")

    with rasterio.open(dsm_path) as dsm_src:
        dsm_crs = dsm_src.crs
        print(f"DSM CRS: {dsm_crs}")

        if sat_crs != dsm_crs:
            print("CRS mismatch, transforming bounds...")

            minx, miny, maxx, maxy = transform_bounds(
                sat_crs, 
                dsm_crs, 
                sat_bounds.left, 
                sat_bounds.bottom, 
                sat_bounds.right, 
                sat_bounds.top
            )
        else:
            minx, miny, maxx, maxy = sat_bounds.left, sat_bounds.bottom, sat_bounds.right, sat_bounds.top

        bbox = box(minx, miny, maxx, maxy)
        geo = [json.loads(json.dumps(bbox.__geo_interface__))]

        print("Cropping the DSM...")
        out_image, out_transform = mask(dsm_src, geo, crop=True)
        
        out_meta = dsm_src.meta.copy()

    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1], 
        "width": out_image.shape[2],  
        "transform": out_transform    
    })

    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(out_image)
    
    print(f"finished! The cropped DSM is saved at {output_path}")


dsm_file = "dsm.tif"       # your big size DSM file
satellite_file = "Google19.tif" # your small size satellite image file
output_file = "cropped_dsm.tif"  # ouput cropped DSM file path

try:
    crop_dsm_to_sat_extent(dsm_file, satellite_file, output_file)
except Exception as e:
    print(f"Error occurred: {e}")

#### Get the metadate json file of Thermal-UAV dataset

In [1]:
import os
import csv
import json
from pathlib import Path
import pandas as pd
def convert_img_info_to_json(modality = 'Thermal'):
    """
    transform the Thermal/Visible data into three JSON files (train, test, valid)
    """
    base_path = Path("Data/Thermal-UAV")
    output_path = Path("Data/metadata")
    output_path.mkdir(parents=True, exist_ok=True)
    
    splits = ['train', 'test', 'valid']
    
    for split in splits:
        split_path = base_path / split
        all_data = []

        changsha_path = split_path / "changsha"
        if not changsha_path.exists():
            continue
            
        # traverse each scene directory under changsha
        for scene_dir in sorted(changsha_path.iterdir()):
            if not scene_dir.is_dir():
                continue
                
            img_info_file = scene_dir / f"{modality}_info.csv"
            if not img_info_file.exists():
                print(f"warning: {img_info_file} does not exist, skip")
                continue
            
            print(f"processing: {img_info_file}")
            
            # read the CSV file
            with open(img_info_file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    filename = row['filename']
                    relative_path = f"./Data/Thermal-UAV/{split}/changsha/{scene_dir.name}/{modality}/{filename}"
                    # Thermal image use DJI M4T drone
                    entry = {
                        "name": relative_path,
                        "lat": float(pd.to_numeric(row['lat'])),
                        "lon": float(pd.to_numeric(row['lon'])),
                        'abs_height': float(pd.to_numeric(row['abs_height'])),
                        "rel_height": float(pd.to_numeric(row['rel_height'])),
                        "pitch": float(pd.to_numeric(row['pitch'])),
                        "yaw": float(pd.to_numeric(row['yaw'])),
                        "roll": float(pd.to_numeric(row['roll'])),
                        "cam_size": 9.83,  # physical length of the diagonal
                        "focal_len": 12.0,  
                        "width": 1280.0,  
                        "height": 1024.0,  
                        "fov": 35.4,  
                        "TargetLatitude": float(pd.to_numeric(row['TargetLatitude'])),
                        "TargetLongitude": float(pd.to_numeric(row['TargetLongitude']))
                    }
                    
                    # # Visible image use DJI M4T drone
                    # entry = {
                    #     "name": relative_path,
                    #     "lat": float(pd.to_numeric(row['lat'])),
                    #     "lon": float(pd.to_numeric(row['lon'])),
                    #     'abs_height': float(pd.to_numeric(row['abs_height'])),
                    #     "rel_height": float(pd.to_numeric(row['rel_height'])),
                    #     "pitch": float(pd.to_numeric(row['pitch'])),
                    #     "yaw": float(pd.to_numeric(row['yaw'])),
                    #     "roll": float(pd.to_numeric(row['roll'])),
                    #     "cam_size": 12.11,  # physical length of the diagonal
                    #     "focal_len": 6.73,  
                    #     "width": 4032.0,  
                    #     "height": 3024.0,  
                    #     "fov": 84.0,  
                    #     "TargetLatitude": float(pd.to_numeric(row['TargetLatitude'])),
                    #     "TargetLongitude": float(pd.to_numeric(row['TargetLongitude']))
                    # }
                    
                    all_data.append(entry)
        
        # save json file
        output_file = output_path / f"{split}_{modality}.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(all_data, f, indent=4, ensure_ascii=False)
        
        print(f"already generated: {output_file}，containing {len(all_data)} records") # if no this print, it is wrong with the path or the csv file, but no error will be raised, so add this print to check if the file is generated successfully
    
    print("\ncompile done!") 

if __name__ == "__main__":
    # modality = 'Visible'
    modality = 'Thermal'
    convert_img_info_to_json(modality)

processing: Data\Thermal-UAV\train\changsha\city2_300_ortho_day\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\city2_300_ortho_night\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\city3_300_ortho_night\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\city4_300_ortho_day\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\city5_300_ortho_night\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\city6_300_ortho_day\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\nudt_300_ortho_night\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\village2_300_ortho_day\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\village2_300_ortho_day&fog\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\village2_300_ortho_night\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\village2_300_ortho_night&fog\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\village3_300_ortho_day\Thermal_inf

#### Deal the pkl file to analysis the results

In [10]:
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

RESULT_ROOT = r'E:\Learning\SCC-Loc\Result\Experiment_20260322_164302\changsha' 
# OUTPUT_DIR = './evaluation_summary'

# standard for evaluation
RETRIEVAL_THRESHOLD = 0.5
TOP_N = 5 # look at the top 10 retrieval results
DIST_THRESHOLDS = [5, 10, 20, 30, 50] # accuracy thresholds for localization recall
PENALTY_VALUE = 1000 # penalty value for failed or missing data

def load_and_parse_pkls(root_dir):
    pkl_files = list(Path(root_dir).rglob("*.pkl"))
    print(f"Found {len(pkl_files)} pkl files in {root_dir}")
    
    data_list = []
    for pkl_path in pkl_files:
        try:
            with open(pkl_path, 'rb') as f:
                data = pickle.load(f)
            
            # retrieval statistics
            pde_list = data.get('PDE', [])
            retrieval_hit = False
            if isinstance(pde_list, (list, np.ndarray)) and len(pde_list) > 0:
                top_n_min_pde = np.min(pde_list[:TOP_N])
                if top_n_min_pde < RETRIEVAL_THRESHOLD:
                    retrieval_hit = True

            # localization error statistics
            err_dict = data.get('detailed_error_dict')
            status = 'Fail'
            h_err, e_alt, e_yaw = PENALTY_VALUE, PENALTY_VALUE, PENALTY_VALUE
            
            if err_dict:
                status = 'Success'
                temp_h_err = err_dict.get('horizontal_error')
                if temp_h_err is None:
                    lat_err = err_dict.get('err_lat')
                    lon_err = err_dict.get('err_lon')
                    if lat_err is not None and lon_err is not None:
                        temp_h_err = np.sqrt(lat_err**2 + lon_err**2)
                
                if temp_h_err is not None and not np.isnan(temp_h_err):
                    h_err = temp_h_err
                
                temp_alt = err_dict.get('err_alt')
                if temp_alt is not None and not np.isnan(temp_alt):
                    e_alt = abs(temp_alt)
                    
                temp_yaw = err_dict.get('err_yaw')
                if temp_yaw is not None and not np.isnan(temp_yaw):
                    e_yaw = abs(temp_yaw)

            # cost time statistics
            time_cost = data.get('total_time', data.get('VG_time_cost', np.nan))

            data_list.append({
                'filename': os.path.basename(data.get('img_path', pkl_path.name)),
                'retrieval_hit': retrieval_hit,
                'horizontal_error': h_err,
                'err_alt': e_alt,
                'err_yaw': e_yaw,
                'status': status,
                'time_cost': time_cost  
            })
        except Exception as e:
            print(f"skipped {pkl_path}: {e}")
        
    return pd.DataFrame(data_list)

def run_report(df):
    if df.empty: return
    total = len(df)
    
    print("\n" + "="*55)
    print(f"📊 Evaluation Summary (Total Samples: {total})")
    print("="*55)

    # --- A. retrieval recall ---
    ret_recall = df['retrieval_hit'].sum() / total * 100
    print(f"1. Retrieval Recall (Top-{TOP_N}, PDE<{RETRIEVAL_THRESHOLD}): {ret_recall:.2f}%")

    # --- B. localization recall ---
    print("\n2. Localization Recall (Full Sample):")
    recall_results = []
    for t in DIST_THRESHOLDS:
        count = ((df['status'] == 'Success') & (df['horizontal_error'] <= t)).sum()
        rate = count / total * 100
        recall_results.append({"Threshold": f"<{t}m", "Success Count": count, "Recall Rate": f"{rate:.2f}%"})
    print(pd.DataFrame(recall_results).to_string(index=False))

    # --- C. error distribution ---
    print(f"\n3. Global Error Distribution (Including Filled with Penalty Value")
    error_stats = df[['horizontal_error', 'err_alt', 'err_yaw']].agg(['mean', 'std', 'median'])
    error_stats.columns = ['horizontal_error(m)', 'err_alt(m)', 'err_yaw(°)']
    print(error_stats.T.to_string())

    # --- [新增] D. cost time statistics ---
    print("\n4. Cost Time Statistics (Time Cost Analysis):")
    # 排除 NaN 值进行计算
    valid_times = df['time_cost'].dropna()
    if not valid_times.empty:
        avg_time = valid_times.mean()
        std_time = valid_times.std()
        med_time = valid_times.median()
        fps = 1.0 / avg_time if avg_time > 0 else 0
        
        print(f"   - mean time (Mean): {avg_time:.4f} seconds/frame")
        print(f"   - standard deviation (Std) : {std_time:.4f} seconds")
        print(f"   - median (Med) : {med_time:.4f} seconds/frame")
        print(f"   - theoretical frame rate (FPS) : {fps:.2f} Hz")
    else:
        print("   - [Warning] did not find valid time_cost data in .pkl files.")
    print("="*55 + "\n")

if __name__ == "__main__":
    df_final = load_and_parse_pkls(RESULT_ROOT)
    run_report(df_final)
    
    # output for ablation study comparison
    # os.makedirs(OUTPUT_DIR, exist_ok=True)
    # df_final.to_excel(os.path.join(OUTPUT_DIR, "detailed_stats.xlsx"), index=False)

Found 2350 pkl files in E:\Learning\SCC-Loc\Result\Experiment_20260322_164302\changsha

📊 Evaluation Summary (Total Samples: 2350)
1. Retrieval Recall (Top-5, PDE<0.5): 96.77%

2. Localization Recall (Full Sample):
Threshold  Success Count Recall Rate
      <5m           1224      52.09%
     <10m           1731      73.66%
     <20m           1888      80.34%
     <30m           1954      83.15%
     <50m           2042      86.89%

3. Global Error Distribution (Including Filled with Penalty Value
                          mean        std     median
horizontal_error(m)  29.643134  79.531132   4.753838
err_alt(m)           23.530080  57.940833  13.521211
err_yaw(°)            5.167174  50.409998   2.092891

4. Cost Time Statistics (Time Cost Analysis):
   - mean time (Mean): 2.2043 seconds/frame
   - standard deviation (Std) : 0.2255 seconds
   - median (Med) : 2.1960 seconds/frame
   - theoretical frame rate (FPS) : 0.45 Hz

